In [ ]:
import ROOT
import numpy as np
import matplotlib.pyplot as plt
import mplhep as hep
import os

hep.style.use(hep.style.ROOT)

# Constants
ALPHA_EM = 1.0 / 137.036
M_E = 0.000511  
E_E = 18.0      
E_P = 275.0     
S = 4.0 * E_E * E_P
Q2_MAX_GEN = 0.01 

# =========================================================================
# CONFIGURATION: UPSILON FAMILY
# =========================================================================
UPSILON_STATES = {
    "1S": {"mass": 9.4603,  "sigma": 0.053, "br_ee": 0.0234, "color": "royalblue"},
    "2S": {"mass": 10.0233, "sigma": 0.056, "br_ee": 0.0191, "color": "crimson"},
    "3S": {"mass": 10.3552, "sigma": 0.059, "br_ee": 0.0218, "color": "forestgreen"}
}

def photon_flux_W(W, Q2_max=0.01):
    y = (W**2) / S
    if y <= 0 or y >= 1: return 0.0
    
    Q2_min = (M_E**2 * y**2) / (1.0 - y)
    if Q2_min >= Q2_max: return 0.0
    
    term1 = (1.0 + (1.0 - y)**2) / y
    log_term = np.log(Q2_max / Q2_min)
    term2 = (2.0 * (1.0 - y)) / y
    
    f_y = (ALPHA_EM / (2.0 * np.pi)) * (term1 * log_term - term2)
    return f_y * ((2.0 * W) / S)

def main():
    filename = os.path.expanduser("~/merged_elecs_test_tuples.root")
    lumi_invfb = 10.0 
    
    try:
        df = ROOT.RDataFrame("UpsilonTree", filename)
    except Exception as e:
        print(f"Error opening file {filename}: {e}")
        return

    W_bins = np.linspace(20, 150, 15)
    bin_centers = (W_bins[:-1] + W_bins[1:]) / 2.0
    bin_width = W_bins[1] - W_bins[0]

    fig, ax = plt.subplots(figsize=(10, 8))

    # HERA 1S Reference Data
    hera_W = [95.0, 175.0]  
    hera_W_err = [35.0, 45.0] 
    hera_sig = [160.0, 321.0]
    hera_err = [70.0, 144.0] 
    
    ax.errorbar(hera_W, hera_sig, xerr=hera_W_err, yerr=hera_err, fmt='^', 
                color='gray', label=r'HERA $\gamma p \rightarrow \Upsilon(1S) p$', markersize=9, linewidth=2)

    for name, props in UPSILON_STATES.items():
        mass_window = 3.0 * props["sigma"]
        mass_min = props["mass"] - mass_window
        mass_max = props["mass"] + mass_window
        
        # 1. Filter Reco Data (Restored pulling the 'weight' branch)
        df_reco = df.Filter(f"reco_mass >= {mass_min} && reco_mass <= {mass_max}") \
                    .Filter("abs(sigma_epz - 36.0) <= 3.6") \
                    .Filter("reco_W >= 20.0")
        data_reco = df_reco.AsNumpy(["reco_W", "weight"])

        # 2. Filter Truth Data 
        df_true = df.Filter(f"true_mass >= {props['mass'] - 0.2} && true_mass <= {props['mass'] + 0.2}") \
                    .Filter("true_W > 20.0")
        data_true = df_true.AsNumpy(["true_W", "weight"])

        if len(data_reco["reco_W"]) == 0:
            print(f"Upsilon({name}): SKIPPED (0 events found)")
            continue

        # Use the ROOT weights exactly as before!
        yield_reco, _ = np.histogram(data_reco["reco_W"], bins=W_bins, weights=data_reco["weight"])
        err_yield_reco = np.sqrt(np.histogram(data_reco["reco_W"], bins=W_bins, weights=data_reco["weight"]**2)[0])
        
        yield_true, _ = np.histogram(data_true["true_W"], bins=W_bins, weights=data_true["weight"])
        
        efficiency = np.zeros_like(yield_true, dtype=float)
        valid_true = yield_true > 0
        efficiency[valid_true] = yield_reco[valid_true] / yield_true[valid_true]

        sigma_gamma_p = np.zeros_like(yield_reco, dtype=float)
        err_sigma_gamma_p = np.zeros_like(err_yield_reco, dtype=float)

        # 3. Unfold Bin-by-Bin 
        for i in range(len(bin_centers)):
            W = bin_centers[i]
            eff = efficiency[i]
            
            if eff > 0 and yield_reco[i] > 0:
                flux = photon_flux_W(W, Q2_max=Q2_MAX_GEN)
                
                # REVERTED: Using the exact conversion factor that matched HERA
                conversion_factor = (lumi_invfb * bin_width * eff * flux * 1e3)
                
                # Keep the relative spacing correct between the 1S, 2S, and 3S
                relative_br_corr = UPSILON_STATES["1S"]["br_ee"] / props["br_ee"]
                
                sigma_gamma_p[i] = (yield_reco[i] / conversion_factor) * relative_br_corr
                err_sigma_gamma_p[i] = (err_yield_reco[i] / conversion_factor) * relative_br_corr

        # 4. Plot
        plot_mask = sigma_gamma_p > 0
        ax.errorbar(bin_centers[plot_mask], sigma_gamma_p[plot_mask], yerr=err_sigma_gamma_p[plot_mask], 
                    fmt='o', color=props["color"], label=rf'ePIC Unfolded $\Upsilon({name})$', markersize=8, linewidth=2)

    # =========================================================================
    # PLOT FORMATTING
    # =========================================================================
    ax.set_xlabel(r"Photon-Proton Center of Mass Energy $W_{\gamma p}$ [GeV]", fontsize=14)
    ax.set_ylabel(r"$\sigma(\gamma p \rightarrow \Upsilon(nS) p)$ [pb]", fontsize=14) 
    
    # Set the y-axis to log scale
    ax.set_yscale('log')

    ax.set_xlim(20, 250)
    
    # CRITICAL: The bottom limit must be > 0. 
    # For cross sections in pb, 1 or 10 is usually a good visual floor.
    # The top limit needs to be expanded to fit the log scale visually.
    ax.set_ylim(1, 1000)
    ax.text(0.04, 0.94, "ePIC Performance", transform=ax.transAxes, fontsize=12, fontweight='bold', ha='left', va='top')
    ax.text(0.04, 0.88, "26.03.1 Campaign", transform=ax.transAxes, fontsize=11, ha='left', va='top')
    ax.text(0.04, 0.82, r"e+p $\Upsilon_{\gamma p}$ 18x275 GeV$^2$", transform=ax.transAxes, fontsize=11, ha='left', va='top')
    ax.text(0.04, 0.76, r"$\mathcal{L}_{proj} = 10$ fb$^{-1}$, $\sqrt{s} = 141$ GeV", transform=ax.transAxes, fontsize=11, ha='left', va='top')

    ax.set_xlim(20, 250)
    ax.set_ylim(0, 600)  
    ax.legend(loc='lower right', fontsize=11, ncol=1)
    
    fig.tight_layout()
    fig.savefig("Upsilon_Family_GammaP.png", dpi=300)
    print("Saved Upsilon_Family_GammaP.png")

if __name__ == "__main__":
    main()